In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

# Move to repo root so all paths work
_p = Path.cwd()
while not (_p / "demos").exists() and _p != _p.parent:
    _p = _p.parent
if (_p / "demos").exists():
    os.chdir(_p)

load_dotenv()
DATA = Path("data")
Path("outputs").mkdir(exist_ok=True)


# Documents

PDFs are images too. Text trapped in a scan, a photo of a posted notice, a 500-page FOIA dump — these are all "images with text in them."

## OCR with an LLM

Send a photo of a document to an LLM. Get structured text back.


**`documents/llm-ocr.py`** — Send an image to an LLM and get structured text extraction back


In [ ]:
from pathlib import Path

import mimetypes
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent

DATA = Path("data")
MODEL = "openai:gpt-5-nano"
IMAGE = DATA / "city.png"

class ExtractedText(BaseModel):
    text: str = Field(description="All visible text, preserving layout")
    text_type: str = Field(description="sign, document, screenshot, handwritten, label, other")
    language: str = Field(description="Primary language of the text")
    confidence: str = Field(description="high, medium, or low")

agent = Agent(MODEL, output_type=ExtractedText)
mime_type, _ = mimetypes.guess_type(str(IMAGE))
image_data = IMAGE.read_bytes()

result = agent.run_sync([
    "Extract all visible text from this image. Preserve layout and reading order.",
    BinaryContent(data=image_data, media_type=mime_type or "image/jpeg"),
])

output = result.output
print(output.text)
print(f"\nType: {output.text_type}  Language: {output.language}  Confidence: {output.confidence}")


## Classify pages visually

Got hundreds of pages from a FOIA? Classify every page as diagram, text, invoice, blank — using CLIP, no API key needed. Then filter to just the ones you want.


**`documents/classify-pages.py`** — Classify pages of a PDF visually (diagram, text, invoice, etc.) using CLIP


In [ ]:
from natural_pdf import PDF

URL = "https://github.com/jsoma/ire25-natural-pdf/raw/refs/heads/main/cia-doc.pdf"

pdf = PDF(URL)
pdf.classify_pages(['diagram', 'text', 'invoice', 'blank'], using='vision')

for page in pdf.pages:
    print(f"Page {page.number}: {page.category} ({page.category_confidence:.2f})")

diagrams = pdf.pages.filter(lambda p: p.category == 'diagram')
print(f"\nFound {len(diagrams)} diagram pages")
diagrams.show(show_category=True)


## Extract structured data

Pull specific fields from a document page. Visual citations show exactly where on the page each answer came from — you can see what the model looked at.


**`documents/extract.py`** — Extract structured data from a PDF page with an LLM, with visual citations


In [ ]:
from pathlib import Path

import os
from openai import OpenAI
from natural_pdf import PDF

URL = "https://github.com/jsoma/natural-pdf/raw/refs/heads/main/pdfs/01-practice.pdf"

client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

pdf = PDF(URL)
page = pdf.pages[0]

fields = ["site", "date", "violation count", "inspection service", "summary", "city", "state"]
results = page.extract(fields, client=client, model="gemini-2.5-flash")
print(results.to_dict())

results.show()


## Extract with a Pydantic schema

Same extraction, but with a Pydantic schema for precise field control. Same pattern as the image demos — define your fields, the model fills them in.


**`documents/extract-pydantic.py`** — Extract structured data from a PDF using a Pydantic schema


In [ ]:
from pathlib import Path

import os
from openai import OpenAI
from pydantic import BaseModel, Field
from natural_pdf import PDF

URL = "https://github.com/jsoma/natural-pdf/raw/refs/heads/main/pdfs/01-practice.pdf"

client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

class ReportInfo(BaseModel):
    inspection_number: str = Field(description="The main report identifier")
    inspection_date: str = Field(description="Date of the inspection")
    inspection_service: str = Field(description="Name of inspection service")
    site: str = Field(description="Name of company inspected")
    summary: str = Field(description="Visit summary")
    city: str
    state: str = Field(description="Full name of state")
    violation_count: int

pdf = PDF(URL)
page = pdf.pages[0]
page.extract(schema=ReportInfo, client=client, model="gemini-2.5-flash")

print(dict(page.extracted()))
